In [1]:
# Purchase Pattern Analytics - Market Basket Analysis (Apriori)


In [3]:
import sys
print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [4]:
import sys
!{sys.executable} -m pip install --user mysql-connector-python

In [5]:
pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import mysql.connector

In [7]:
import sys
print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [8]:
!pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable


In [9]:
import mysql.connector

print("MySQL Connector is working!")

MySQL Connector is working!


In [10]:
pip install flask

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
import mysql.connector

db = mysql.connector.connect(
    host="18.136.157.135",
    user="dm_team16",
    password="2o_hihiFeTRE"
)

cursor = db.cursor()
cursor.execute("SHOW DATABASES")

for i in cursor:
    print(i)

('information_schema',)
('Automobile Insurance',)
('Fast food analysis',)
('OnlineBookstore',)
('US_Shoppers',)
('customer segmentation',)
('datamites_sql',)
('dell-company',)
('mysql',)
('onlinebookstore',)
('performance_schema',)
('project_automobile_insurance',)
('project_banking',)
('project_crop_production_india',)
('project_csat_analytics',)
('project_customer_segmentation',)
('project_food',)
('project_health_data',)
('project_hr_analytics',)
('project_itsm',)
('project_job_market_analysis',)
('project_medical_data_history',)
('project_movie_database',)
('project_orders',)
('project_profit_analysis',)
('project_purchase_pattern_analysis',)
('project_sales',)
('project_service_data',)
('project_shopping Data',)
('project_shopping_trends',)
('project_snack_analysis',)
('project_supply_chain_data',)
('project_telecom',)
('sys',)
('test',)


In [1]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
conn = mysql.connector.connect(
    host="18.136.157.135",
    user="dm_team16",
    password="2o_hihiFeTRE",
    database="project_purchase_pattern_analysis"
)

print("Database Connected Successfully")

Database Connected Successfully


In [14]:
cursor = conn.cursor()
cursor.execute("SHOW TABLES")

for table in cursor:
    print(table)

('mytable',)


In [15]:
df = pd.read_sql("SELECT * FROM mytable", conn)
df.head()

C:\Users\s\AppData\Local\Temp\ipykernel_1568\3142224628.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM mytable", conn)


,BillNo,Itemname,Quantity,Present_Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom
1,536365,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom


In [16]:
df.columns

Index(['BillNo', 'Itemname', 'Quantity', 'Present_Date', 'Price', 'CustomerID',
       'Country'],
      dtype='object')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522064 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   BillNo        522064 non-null  object 
 1   Itemname      522064 non-null  object 
 2   Quantity      522064 non-null  int64  
 3   Present_Date  522064 non-null  object 
 4   Price         522064 non-null  float64
 5   CustomerID    522064 non-null  object 
 6   Country       522064 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 27.9+ MB


In [18]:
df = df.rename(columns={
    'BillNo': 'InvoiceNo',
    'Itemname': 'Description',
    'Quantity': 'Quantity'
})

df.head()


,InvoiceNo,Description,Quantity,Present_Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom
1,536365,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom


In [19]:
# Remove missing Invoice or Product names
df = df.dropna(subset=['InvoiceNo', 'Description'])

# Convert InvoiceNo to string
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# Remove cancelled invoices (if any)
df = df[~df['InvoiceNo'].str.startswith('C')]

# Keep only positive quantity
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df = df[df['Quantity'] > 0]

df.head()


,InvoiceNo,Description,Quantity,Present_Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom
1,536365,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom


In [20]:
transactions = (
    df.groupby('InvoiceNo')['Description']
      .apply(list)
      .tolist()
)

transactions[:5]


[['WHITE HANGING HEART T-LIGHT HOLDER',
  'WHITE METAL LANTERN',
  'CREAM CUPID HEARTS COAT HANGER',
  'KNITTED UNION FLAG HOT WATER BOTTLE',
  'RED WOOLLY HOTTIE WHITE HEART.',
  'SET 7 BABUSHKA NESTING BOXES',
  'GLASS STAR FROSTED T-LIGHT HOLDER'],
 ['HAND WARMER UNION JACK', 'HAND WARMER RED POLKA DOT'],
 ['ASSORTED COLOUR BIRD ORNAMENT',
  "POPPY'S PLAYHOUSE BEDROOM",
  "POPPY'S PLAYHOUSE KITCHEN",
  'FELTCRAFT PRINCESS CHARLOTTE DOLL',
  'IVORY KNITTED MUG COSY',
  'BOX OF 6 ASSORTED COLOUR TEASPOONS',
  'BOX OF VINTAGE JIGSAW BLOCKS',
  'BOX OF VINTAGE ALPHABET BLOCKS',
  'HOME BUILDING BLOCK WORD',
  'LOVE BUILDING BLOCK WORD',
  'RECIPE BOX WITH METAL HEART',
  'DOORMAT NEW ENGLAND'],
 ['JAM MAKING SET WITH JARS',
  'RED COAT RACK PARIS FASHION',
  'YELLOW COAT RACK PARIS FASHION',
  'BLUE COAT RACK PARIS FASHION'],
 ['BATH BUILDING BLOCK WORD']]

In [24]:
!pip install mlxtend

Defaulting to user installation because normal site-packages is not writeable


In [25]:
from mlxtend.preprocessing import TransactionEncoder

In [26]:
# one-hot encoding
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_array, columns=te.columns_)

basket.head()

,,*Boombox Ipod Classic,*USB Office Mirror Ball,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [27]:
# Count how many times each product appears
item_freq = basket.sum()

# Keep only products bought at least 100 times
basket_reduced = basket[item_freq[item_freq >= 100].index]

basket_reduced.shape


(20327, 1532)

In [28]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    basket_reduced,
    min_support=0.02,   # increase support
    use_colnames=True
)

frequent_itemsets.head()


,support,itemsets
0,0.029124,()
1,0.022482,(3 STRIPEY MICE FELTCRAFT)
2,0.022827,(4 TRADITIONAL SPINNING TOPS)
3,0.046342,(6 RIBBONS RUSTIC CHARM)
4,0.020515,(60 CAKE CASES DOLLY GIRL DESIGN)


In [29]:
frequent_itemsets.sort_values(by='support', ascending=False).head(10)

,support,itemsets
263,0.108329,(WHITE HANGING HEART T-LIGHT HOLDER)
98,0.101540,(JUMBO BAG RED RETROSPOT)
191,0.093669,(REGENCY CAKESTAND 3 TIER)
152,0.081468,(PARTY BUNTING)
123,0.075810,(LUNCH BAG RED RETROSPOT)
16,0.070399,(ASSORTED COLOUR BIRD ORNAMENT)
210,0.066217,(SET OF 3 CAKE TINS PANTRY DESIGN)
144,0.062921,(PACK OF 72 RETROSPOT CAKE CASES)
116,0.061987,(LUNCH BAG BLACK SKULL.)
134,0.060609,(NATURAL SLATE HEART CHALKBOARD)


In [30]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=1
)

rules[['antecedents','consequents','support','confidence','lift']].head()


,antecedents,consequents,support,confidence,lift
0,(PACK OF 72 RETROSPOT CAKE CASES),(60 TEATIME FAIRY CAKE CASES),0.021400,0.340109,8.663415
1,(60 TEATIME FAIRY CAKE CASES),(PACK OF 72 RETROSPOT CAKE CASES),0.021400,0.545113,8.663415
2,(ALARM CLOCK BAKELIKE PINK),(ALARM CLOCK BAKELIKE GREEN),0.020219,0.541502,11.382741
3,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE PINK),0.020219,0.425026,11.382741
4,(ALARM CLOCK BAKELIKE RED),(ALARM CLOCK BAKELIKE GREEN),0.031141,0.616358,12.956273


In [31]:
strong_rules = rules[
    (rules['confidence'] >= 0.6) &
    (rules['lift'] > 1)
]

strong_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4,(ALARM CLOCK BAKELIKE RED),(ALARM CLOCK BAKELIKE GREEN),0.050524,0.047572,0.031141,0.616358,12.956273,1.0,0.028737,2.482597,0.971923,0.465099,0.597196,0.635480
5,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.047572,0.050524,0.031141,0.654602,12.956273,1.0,0.028737,2.748932,0.968911,0.465099,0.636222,0.635480
11,(CHARLOTTE BAG PINK POLKADOT),(RED RETROSPOT CHARLOTTE BAG),0.036306,0.050573,0.025582,0.704607,13.932439,1.0,0.023746,3.214115,0.963195,0.417335,0.688872,0.605222
18,(DOLLY GIRL LUNCH BOX),(SPACEBOY LUNCH BOX),0.040832,0.043489,0.025828,0.632530,14.544615,1.0,0.024052,2.602964,0.970890,0.441548,0.615823,0.613211
20,(DOTCOM POSTAGE),(JUMBO BAG RED RETROSPOT),0.034831,0.101540,0.023860,0.685028,6.746400,1.0,0.020323,2.852510,0.882511,0.212068,0.649432,0.460004
22,(GARDENERS KNEELING PAD KEEP CALM),(GARDENERS KNEELING PAD CUP OF TEA),0.044522,0.037094,0.026762,0.601105,16.205120,1.0,0.025111,2.413935,0.982012,0.487892,0.585739,0.661295
23,(GARDENERS KNEELING PAD CUP OF TEA),(GARDENERS KNEELING PAD KEEP CALM),0.037094,0.044522,0.026762,0.721485,16.205120,1.0,0.025111,3.430621,0.974436,0.487892,0.708508,0.661295
24,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.047966,0.036208,0.029763,0.620513,17.137451,1.0,0.028027,2.539722,0.989091,0.547016,0.606256,0.721262
25,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.036208,0.047966,0.029763,0.822011,17.137451,1.0,0.028027,5.348834,0.977024,0.547016,0.813043,0.721262
28,(ROSES REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.049835,0.047966,0.035962,0.721619,15.044460,1.0,0.033572,3.419896,0.982493,0.581543,0.707593,0.735681


In [33]:
strong_rules.to_csv("apriori_results.csv", index=False)
print("Results Saved Successfully")

Results Saved Successfully


In [34]:
# Top 10 Frequent itemsets
frequent_itemsets.sort_values(by='support', ascending=False).head(10)

,support,itemsets
263,0.108329,(WHITE HANGING HEART T-LIGHT HOLDER)
98,0.101540,(JUMBO BAG RED RETROSPOT)
191,0.093669,(REGENCY CAKESTAND 3 TIER)
152,0.081468,(PARTY BUNTING)
123,0.075810,(LUNCH BAG RED RETROSPOT)
16,0.070399,(ASSORTED COLOUR BIRD ORNAMENT)
210,0.066217,(SET OF 3 CAKE TINS PANTRY DESIGN)
144,0.062921,(PACK OF 72 RETROSPOT CAKE CASES)
116,0.061987,(LUNCH BAG BLACK SKULL.)
134,0.060609,(NATURAL SLATE HEART CHALKBOARD)


In [35]:
strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False).head(10)

,antecedents,consequents,support,confidence,lift
150,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.025336,0.704514,19.457423
155,(PINK REGENCY TEACUP AND SAUCER),"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",0.025336,0.699728,19.457423
152,"(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.025336,0.903509,18.836536
24,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.029763,0.620513,17.137451
25,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.029763,0.822011,17.137451
151,"(PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY...",(ROSES REGENCY TEACUP AND SAUCER),0.025336,0.851240,17.081095
22,(GARDENERS KNEELING PAD KEEP CALM),(GARDENERS KNEELING PAD CUP OF TEA),0.026762,0.601105,16.205120
23,(GARDENERS KNEELING PAD CUP OF TEA),(GARDENERS KNEELING PAD KEEP CALM),0.026762,0.721485,16.205120
139,(PINK REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.028042,0.774457,15.540353
29,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.035962,0.749744,15.044460
